# Desvanecimiento del gradiente.

En esta libreta exploraremos dos aspectos fundamentales del aprendizaje profundo. Primero, experimentaremos con el problema del desvanecimiento de gradientes para entender por qué algunas redes profundas no logran entrenar correctamente y cómo técnicas modernas como ReLU, inicialización He, batch normalization y clipping ayudan a resolverlo. 

Cargamos librerias y dataset. En esta ocasión usaremos el dataset MNIST y la librería PyTorch.



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms


SEED = 42
torch.manual_seed(SEED)


if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

transform = transforms.Compose([transforms.ToTensor()])
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo: ", device)



Definimos un modelo y una función de entrenamiento. Tomate un tiempo para analizar el siguiente código y familiarizarte poco a poco con PyTorch. Si tienes dudas de alguna línea pregunta al profesor.

In [ ]:
class Net1(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        input_size = 28*28  # capa de entrada
        hidden_size = 128
        for _ in range(6):  # 6 capas densas
            layer = nn.Linear(input_size, hidden_size)
            nn.init.normal_(layer.weight, mean=0.0, std=0.01)
            nn.init.constant_(layer.bias, 0.0)

            layers.append(layer)
            layers.append(nn.Sigmoid())
            input_size = hidden_size
        layers.append(nn.Linear(hidden_size, 10))  # capa de salida
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

def train(model, optimizer, criterion, epochs=3, clip=False):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        grad_means = []         # para guardar promedios por batch (global)
        grad_first_layer = []   # para la primera capa
        grad_last_layer = []    # para la última capa

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()

            batch_grad_means = []
            params = list(model.named_parameters())
            first_name, first_param = params[0]   # primera capa
            last_name, last_param = params[-2]    # última capa (el penúltimo suele ser weight, el último es bias)

            for name, param in params:
                if param.grad is not None:
                    gmean = param.grad.abs().mean().item()
                    batch_grad_means.append(gmean)
                    if name == first_name:
                        grad_first_layer.append(gmean)
                    if name == last_name:
                        grad_last_layer.append(gmean)

            grad_means.append(sum(batch_grad_means)/len(batch_grad_means))
            # clipping opcional
            if clip:
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(trainloader):.4f}")
        print(f"  Grad mean (global): {sum(grad_means)/len(grad_means):.6f}")
        print(f"  Grad mean (primera capa): {sum(grad_first_layer)/len(grad_first_layer):.6f}")
        print(f"  Grad mean (última capa): {sum(grad_last_layer)/len(grad_last_layer):.6f}")


Antes de correr el entrenamiento en la siguiente celda, calcula manualmente
el valor esperado de la pérdida en el caso especial en que el modelo
no esté aprendiendo nada, es decir, que adivine al azar asignando la
misma probabilidad a todas las clases.

Este valor servirá como referencia: si tu modelo se queda cerca de este
número, significa que no está aprendiendo.

Recuerda que estamos usando la función de pérdida entropía cruzada (cross-entropy):

$$
L = - \sum_{i=1}^{C} y_i \, \log(p_i)
$$

donde:

- C = número de clases  
- y_i = etiqueta verdadera en formato one-hot  
- p_i = probabilidad predicha para la clase i  

---

### Ejemplo (con 5 clases en lugar de 10)
Supongamos un problema de clasificación con **C = 5** clases.  
Si el modelo adivina, asigna:  
$$
p_i = \frac{1}{5} = 0.2
$$

Entonces, para la clase correcta:

$$
L = - \log(0.2) = \log(5) \approx 1.609
$$

Esto significa que en un problema con 5 clases, un modelo que adivina tendría
una pérdida cercana a 1.61.  

---

Ahora haz el mismo cálculo para MNIST (C = 10)
y compáralo con el valor que verás al entrenar.


In [ ]:
model_bad = Net1().to(device)
optimizer = optim.SGD(model_bad.parameters(), lr=0.1)  # lr alto para ver problemas
criterion = nn.CrossEntropyLoss()

print("Entrenando modelo con Sigmoid + init normal...")
train(model_bad, optimizer, criterion, epochs=3)

Responde las preguntas:

1. ¿Cómo evoluciona el valor de la pérdida en cada época? ¿Disminuye, se mantiene estable o incluso aumenta?
2. ¿El valor de pérdida que arroja el modelo es cercano al valor esperado cuando el modelo solo adivina?
3. ¿Qué interpretación de das a ese resultado?
4. ¿Por qué crees que el modelo no está aprendiendo (o aprendiendo tan poco)? Piensa en activaciones, inicialización, normalización, clipping, etc.
5. ¿Qué técnicas o cambios podrías aplicar para que el modelo aprenda mejor? Pista: funciones de activación, inicialización, normalización, clipping, etc.


Ahora modifica el modelo base aplicando las técnicas que consideres convenientes. Algunos integrantes del equipo pueden experimentar cambiando la función de pérdida por otra distinta, otros pueden probar diferentes métodos de inicialización de pesos (por ejemplo, Xavier o He/Kaiming), y otros pueden añadir capas de batch normalization después de las capas ocultas. La idea es que cada uno explore una técnica distinta y, al final, combinen todas estas mejoras en un solo modelo para observar si el entrenamiento y los resultados se ven favorecidos en comparación con el modelo original.

Nota: La práctica estándar es colocar las capas en el siguiente orden: Linear -> BatchNorm -> ReLu

In [ ]:
class Net2(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        input_size = 28*28
        hidden_size = 128
        for _ in range(6):
            layer = nn.Linear(input_size, hidden_size)
            # Modifica aquí la instruccion para inicializar los pesos de layer
            nn.init.kaiming_normal_(layer.weight)
            nn.init.constant_(layer.bias, 0.0)

            layers.append(layer)
            # Coloca aquí la instrucion para agregar una capa de normalización de lote (debe ser nn.BatchNorm1d)
            # Reemplaza la función de activación en la siguiente línea por aquella que desees probar.
            #layers.append(nn.Sigmoid())
            input_size = hidden_size
        layers.append(nn.Linear(hidden_size, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.view(x.size(0), -1))


In [ ]:
print("Entrenando modelo Net2")
model_good = Net2().to(device)
optimizer = optim.SGD(model_good.parameters(), lr=0.1)
train(model_good, optimizer, criterion, epochs=3, clip=True)  # con clipping